In [1]:
from google.cloud import bigquery
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import xgboost as xgb

client = bigquery.Client(project="YOUR_PROJECT_ID")

In [8]:
from google.cloud import bigquery

client = bigquery.Client(project="modelraja")

query = """
SELECT * FROM `modelraja.MODELS_DATASET.VW_top_gainers_losers_training_data` 
"""

df = client.query(query).to_dataframe(create_bqstorage_client=True)
print(df.head())

  Ticker  Stock_Price Stock_Snapshot_Date Performance_Week  \
0   NCMI         3.25          2026-04-06            6.91%   
1    CVU         3.33          2026-04-06          -16.54%   
2   KTOS        74.09          2026-04-06            2.99%   
3    IVZ        22.88          2026-04-06           -1.38%   
4   CVGI         4.27          2026-04-06           21.65%   

  Performance_Half_Year Volatility_Week next_day_Snapshot_Date  \
0               -24.42%           3.61%             2026-04-07   
1                31.78%          10.87%             2026-04-07   
2               -22.04%           8.61%             2026-04-07   
3                -3.09%           3.44%             2026-04-07   
4               142.61%           8.96%             2026-04-07   

   eod_nextday_High  eod_nextday_Low         Earnings_Date  ...  \
0              3.33             3.17  2/26/2026 4:30:00 PM  ...   
1              3.41             3.11                        ...   
2             72.90          

In [ ]:
# ── FinBERT Sentiment Feature Engineering ──────────────────────────────────
import re
import torch
import pandas as pd
from transformers import pipeline as hf_pipeline
from google.cloud import bigquery as bq


def extract_useful_slug(url: str):
    """Return cleaned slug text from a URL, or None if segment is opaque/too short."""
    url = url.strip()
    path = url.split("?")[0].rstrip("/")
    segment = path.split("/")[-1]
    segment = re.sub(r'\.[a-z]{2,4}$', '', segment)
    # Skip pure UUID/hash segments (e.g. 679fc12e-b2e4-3f7f-...)
    if re.match(r'^[0-9a-f]{8}-[0-9a-f]{4}', segment):
        return None
    text = segment.replace("-", " ").replace("_", " ").strip()
    if len(text) < 15:
        return None
    return text


# Load sentiment table
sent_query = """
    SELECT Ticker, DATE(snapshot_date_time) AS snapshot_date, urls
    FROM `modelraja.UPCOMING_EARNINGS_DATASET.finviz_news_sentiment_score_3pm`
"""
sent_df = bq.Client(project="modelraja").query(sent_query).to_dataframe()

# Explode comma-separated URLs and extract useful slug text
sent_df["url_list"] = sent_df["urls"].str.split(",")
sent_df = sent_df.explode("url_list")
sent_df["slug_text"] = sent_df["url_list"].apply(
    lambda u: extract_useful_slug(str(u)) if pd.notna(u) else None
)
sent_df = sent_df[sent_df["slug_text"].notna()].copy()

print(f"Slugs to score: {len(sent_df)}")

# Run FinBERT on GPU (falls back to CPU if no GPU)
device = 0 if torch.cuda.is_available() else -1
finbert = hf_pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    device=device,
    top_k=None,  # return all 3 label probabilities
)

BATCH = 64
texts = sent_df["slug_text"].tolist()
results = []
for i in range(0, len(texts), BATCH):
    results.extend(finbert(texts[i : i + BATCH], truncation=True, max_length=128))

# Write scores back into sent_df
label_map = {
    "positive": "finbert_positive",
    "negative": "finbert_negative",
    "neutral":  "finbert_neutral",
}
for col in label_map.values():
    sent_df[col] = float("nan")

for row_scores, idx in zip(results, sent_df.index):
    for item in row_scores:
        col = label_map.get(item["label"].lower())
        if col:
            sent_df.at[idx, col] = item["score"]

for col in label_map.values():
    sent_df[col] = pd.to_numeric(sent_df[col], errors="coerce")

# Aggregate mean scores per Ticker + date
finbert_agg = (
    sent_df.groupby(["Ticker", "snapshot_date"])[list(label_map.values())]
    .mean()
    .reset_index()
)
finbert_agg["finbert_compound"] = (
    finbert_agg["finbert_positive"] - finbert_agg["finbert_negative"]
)

# LEFT JOIN onto main df (NaN rows handled by existing SimpleImputer(median))
df["_snap_date"] = pd.to_datetime(df["Stock_Snapshot_Date"], errors="coerce").dt.date
finbert_agg["snapshot_date"] = pd.to_datetime(finbert_agg["snapshot_date"]).dt.date
df = df.merge(
    finbert_agg.rename(columns={"snapshot_date": "_snap_date"}),
    on=["Ticker", "_snap_date"],
    how="left",
).drop(columns=["_snap_date"])

print(
    f"FinBERT joined. Coverage: {df['finbert_compound'].notna().sum()}/{len(df)} rows"
)
print(
    df[["Ticker", "Stock_Snapshot_Date",
        "finbert_positive", "finbert_negative",
        "finbert_neutral", "finbert_compound"]].head()
)


In [9]:
df.head()

,Ticker,Stock_Price,Stock_Snapshot_Date,Performance_Week,Performance_Half_Year,Volatility_Week,next_day_Snapshot_Date,eod_nextday_High,eod_nextday_Low,Earnings_Date,...,strike_to_close_price_gap,days_to_earnings,todays_range,days_earnings_to_expiry,days_to_options_expiry,Target_Price,Relative_Volume,Volume,Market_cap,Calls_OpenInterest
0,NCMI,3.25,2026-04-06,6.91%,-24.42%,3.61%,2026-04-07,3.33,3.17,2/26/2026 4:30:00 PM,...,0.75,-39,0.31,50.0,11.0,5.6,3.46,1467869,302.72,5.0
1,CVU,3.33,2026-04-06,-16.54%,31.78%,10.87%,2026-04-07,3.41,3.11,,...,0.83,<NA>,0.32,NaN,11.0,5.5,0.97,124294,43.99,178.0
2,KTOS,74.09,2026-04-06,2.99%,-22.04%,8.61%,2026-04-07,72.90,69.80,2/23/2026 4:30:00 PM,...,0.59,-42,4.20,46.0,4.0,119.0,0.93,4140202,13836.88,15.0
3,IVZ,22.88,2026-04-06,-1.38%,-3.09%,3.44%,2026-04-07,22.92,21.82,4/28/2026 8:30:00 AM,...,-0.12,22,1.37,-11.0,11.0,29.5,1.92,11084943,10151.25,311.0
4,CVGI,4.27,2026-04-06,21.65%,142.61%,8.96%,2026-04-07,4.39,3.99,3/10/2026 4:30:00 PM,...,-0.73,-27,0.77,38.0,11.0,4.5,1.00,1106923,156.44,267.0


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10054 entries, 0 to 10053
Data columns (total 33 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Ticker                               10054 non-null  object 
 1   Stock_Price                          10054 non-null  float64
 2   Stock_Snapshot_Date                  10054 non-null  object 
 3   Performance_Week                     10054 non-null  object 
 4   Performance_Half_Year                10052 non-null  object 
 5   Volatility_Week                      10054 non-null  object 
 6   next_day_Snapshot_Date               10054 non-null  object 
 7   eod_nextday_High                     9888 non-null   float64
 8   eod_nextday_Low                      9888 non-null   float64
 9   Earnings_Date                        10029 non-null  object 
 10  Average_True_Range                   10054 non-null  float64
 11  Change                      

In [11]:
df.columns

Index(['Ticker', 'Stock_Price', 'Stock_Snapshot_Date', 'Performance_Week',
       'Performance_Half_Year', 'Volatility_Week', 'next_day_Snapshot_Date',
       'eod_nextday_High', 'eod_nextday_Low', 'Earnings_Date',
       'Average_True_Range', 'Change', 'Call_Option_Expiry_Date',
       'Call_Option_Strike', 'Put_Option_Strike', 'Call_Option_Price',
       'Put_Option_Expiry_Date', 'Put_Option_Price',
       'price_value_change_nextday_day_high',
       'price_value_change_nextday_day_low', 'nextday_range',
       'price_value_delta_above_put_flag', 'price_value_delta_above_call_flag',
       'strike_to_close_price_gap', 'days_to_earnings', 'todays_range',
       'days_earnings_to_expiry', 'days_to_options_expiry', 'Target_Price',
       'Relative_Volume', 'Volume', 'Market_cap', 'Calls_OpenInterest'],
      dtype='object')

In [12]:
# Drop specified columns from existing dataframe `df`
# Keep put target for this notebook; drop call target instead.
# Rename expiry date to a generic name before dropping the other side's expiry.
df.rename(columns={"Put_Option_Expiry_Date": "Option_Expiry_Date"}, inplace=True)

cols_to_drop = [
    "Call_Option_Expiry_Date",
    "price_value_delta_above_call_flag",
    "next_day_Snapshot_Date",
    "nextday_range",
    "price_value_change_nextday_day_high",
    "price_value_change_nextday_day_low",
    "eod_nextday_High",
    "eod_nextday_Low",
    "Put_Option_Strike",
]
found = [c for c in cols_to_drop if c in df.columns]
if found:
    df.drop(columns=found, inplace=True)
print("Dropped columns:", found)
print("Remaining columns:", df.columns.tolist())


Dropped columns: ['Call_Option_Expiry_Date', 'price_value_delta_above_call_flag', 'next_day_Snapshot_Date', 'nextday_range', 'price_value_change_nextday_day_high', 'price_value_change_nextday_day_low', 'eod_nextday_High', 'eod_nextday_Low', 'Put_Option_Strike']
Remaining columns: ['Ticker', 'Stock_Price', 'Stock_Snapshot_Date', 'Performance_Week', 'Performance_Half_Year', 'Volatility_Week', 'Earnings_Date', 'Average_True_Range', 'Change', 'Call_Option_Strike', 'Call_Option_Price', 'Option_Expiry_Date', 'Put_Option_Price', 'price_value_delta_above_put_flag', 'strike_to_close_price_gap', 'days_to_earnings', 'todays_range', 'days_earnings_to_expiry', 'days_to_options_expiry', 'Target_Price', 'Relative_Volume', 'Volume', 'Market_cap', 'Calls_OpenInterest']


In [13]:
df.columns

Index(['Ticker', 'Stock_Price', 'Stock_Snapshot_Date', 'Performance_Week',
       'Performance_Half_Year', 'Volatility_Week', 'Earnings_Date',
       'Average_True_Range', 'Change', 'Call_Option_Strike',
       'Call_Option_Price', 'Option_Expiry_Date', 'Put_Option_Price',
       'price_value_delta_above_put_flag', 'strike_to_close_price_gap',
       'days_to_earnings', 'todays_range', 'days_earnings_to_expiry',
       'days_to_options_expiry', 'Target_Price', 'Relative_Volume', 'Volume',
       'Market_cap', 'Calls_OpenInterest'],
      dtype='object')

In [14]:
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

TARGET_COL = "price_value_delta_above_put_flag"
ID_COLS = ["Ticker", "Stock_Snapshot_Date", "Earnings_Date", "Option_Expiry_Date"]

working_df = df.copy()
print("Training target:", TARGET_COL)
print("Rows:", len(working_df))
print("Columns:", working_df.columns.tolist())


Training target: price_value_delta_above_put_flag
Rows: 10054
Columns: ['Ticker', 'Stock_Price', 'Stock_Snapshot_Date', 'Performance_Week', 'Performance_Half_Year', 'Volatility_Week', 'Earnings_Date', 'Average_True_Range', 'Change', 'Call_Option_Strike', 'Call_Option_Price', 'Option_Expiry_Date', 'Put_Option_Price', 'price_value_delta_above_put_flag', 'strike_to_close_price_gap', 'days_to_earnings', 'todays_range', 'days_earnings_to_expiry', 'days_to_options_expiry', 'Target_Price', 'Relative_Volume', 'Volume', 'Market_cap', 'Calls_OpenInterest']


In [15]:
filtered_df = working_df[working_df[TARGET_COL].notna()].copy()
if filtered_df.empty:
    raise ValueError(f"No non-null rows found for target: {TARGET_COL}")

# Time-based split: year 2026 is test; earlier years are train/validation.
filtered_df["Stock_Snapshot_Date"] = pd.to_datetime(filtered_df["Stock_Snapshot_Date"], errors="coerce")
filtered_df = filtered_df[filtered_df["Stock_Snapshot_Date"].notna()].copy()
filtered_df["snapshot_year"] = filtered_df["Stock_Snapshot_Date"].dt.year

filtered_df["split"] = "test"
non_test_mask = filtered_df["snapshot_year"] < 2026

# Deterministic split for non-test rows only.
key_ticker = filtered_df.loc[non_test_mask, "Ticker"].fillna("").astype(str)
key_date = filtered_df.loc[non_test_mask, "Stock_Snapshot_Date"].dt.strftime("%Y-%m-%d")
split_key = key_ticker + "|" + key_date
bucket = (pd.util.hash_pandas_object(split_key, index=False).astype("uint64") % 100).astype(int)

filtered_df.loc[non_test_mask, "split"] = "validation"
filtered_df.loc[non_test_mask, "split"] = ["train" if b < 85 else "validation" for b in bucket]

print("Split counts:\n", filtered_df["split"].value_counts(dropna=False))
print("Year counts:\n", filtered_df["snapshot_year"].value_counts(dropna=False).sort_index())

# Convert percentage-like columns to numeric fraction (e.g., "7.5%" -> 0.075)
percent_cols = ["Performance_Week", "Performance_Half_Year", "Volatility_Week", "Change"]
for col in percent_cols:
    if col in filtered_df.columns:
        filtered_df[col] = (
            filtered_df[col]
            .astype(str)
            .str.replace("%", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        filtered_df[col] = pd.to_numeric(filtered_df[col], errors="coerce") / 100.0
# Normalize Market_Cap naming and type
if "Market_Cap" not in filtered_df.columns and "Market_cap" in filtered_df.columns:
    filtered_df["Market_Cap"] = filtered_df["Market_cap"]
if "Market_Cap" in filtered_df.columns:
    filtered_df["Market_Cap"] = (
        filtered_df["Market_Cap"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    filtered_df["Market_Cap"] = pd.to_numeric(filtered_df["Market_Cap"], errors="coerce")


drop_cols = ["price_value_delta_above_call_flag", "price_value_delta_above_put_flag", "split", "snapshot_year"] + ID_COLS
feature_cols = [c for c in filtered_df.columns if c not in drop_cols]

train_df = filtered_df[filtered_df["split"] == "train"].copy()
val_df = filtered_df[filtered_df["split"] == "validation"].copy()
test_df = filtered_df[filtered_df["split"] == "test"].copy()

if train_df.empty or val_df.empty or test_df.empty:
    raise ValueError("One or more splits are empty. Check source data/year distribution.")

X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET_COL].copy()
X_val = val_df[feature_cols].copy()
y_val = val_df[TARGET_COL].copy()
X_test = test_df[feature_cols].copy()
y_test = test_df[TARGET_COL].copy()

id_val = val_df[ID_COLS].copy() if all(c in val_df.columns for c in ID_COLS) else pd.DataFrame(index=val_df.index)
id_test = test_df[ID_COLS].copy() if all(c in test_df.columns for c in ID_COLS) else pd.DataFrame(index=test_df.index)

numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number", "bool"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_features),
    ]
)

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)
y_test_enc = le.transform(y_test)

xgb_clf = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

model_xgb = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", xgb_clf),
])

model_xgb.fit(X_train, y_train_enc)

# Baseline validation evaluation
pred_val_base_enc = model_xgb.predict(X_val)
pred_val_base = le.inverse_transform(pred_val_base_enc)

print("Base Validation Accuracy:", accuracy_score(y_val, pred_val_base))
print("Base Validation Classification Report:\n", classification_report(y_val, pred_val_base))
print("Base Validation Confusion Matrix:\n", confusion_matrix(y_val, pred_val_base))

# Hyperparameter tuning (on train only)
param_dist = {
    "classifier__n_estimators": [100, 200, 300, 500],
    "classifier__max_depth": [3, 5, 6, 8],
    "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "classifier__subsample": [0.6, 0.8, 1.0],
    "classifier__colsample_bytree": [0.6, 0.8, 1.0],
    "classifier__gamma": [0, 1, 5],
}

search = RandomizedSearchCV(
    estimator=model_xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring="f1_macro",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train_enc)
best_model = search.best_estimator_

# Validation metrics for tuned model
pred_val_tuned_enc = best_model.predict(X_val)
pred_val_tuned = le.inverse_transform(pred_val_tuned_enc)

print("Best params:", search.best_params_)
print("Best CV score:", search.best_score_)
print("Tuned Validation Accuracy:", accuracy_score(y_val, pred_val_tuned))
print("Tuned Validation Classification Report:\n", classification_report(y_val, pred_val_tuned))
print("Tuned Validation Confusion Matrix:\n", confusion_matrix(y_val, pred_val_tuned))

# Final test metrics
pred_test_tuned_enc = best_model.predict(X_test)
pred_test_tuned = le.inverse_transform(pred_test_tuned_enc)

print("Tuned Test Accuracy:", accuracy_score(y_test, pred_test_tuned))
print("Tuned Test Classification Report:\n", classification_report(y_test, pred_test_tuned))
print("Tuned Test Confusion Matrix:\n", confusion_matrix(y_test, pred_test_tuned))

# Save test predictions
out_df = id_test.copy()
out_df["target_type"] = TARGET_COL
out_df["split"] = "test"
out_df["actual"] = y_test
out_df["predicted"] = pred_test_tuned

if hasattr(best_model.named_steps["classifier"], "predict_proba"):
    pred_proba_test = best_model.predict_proba(X_test)
    for i, cls in enumerate(le.classes_):
        out_df[f"prob_{cls}"] = pred_proba_test[:, i]

pred_path = f"xgboost_tuned_predictions_{TARGET_COL}.csv"
out_df.to_csv(pred_path, index=False)
print(f"Saved tuned test predictions to {pred_path}")
print(out_df.head(20))





Split counts:
 split
train         6869
test          1869
validation    1316
Name: count, dtype: int64
Year counts:
 snapshot_year
2024    3424
2025    4761
2026    1869
Name: count, dtype: int64
Base Validation Accuracy: 0.7401215805471124
Base Validation Classification Report:
               precision    recall  f1-score   support

        Beat       0.61      0.48      0.54       412
      NoBeat       0.78      0.86      0.82       904

    accuracy                           0.74      1316
   macro avg       0.70      0.67      0.68      1316
weighted avg       0.73      0.74      0.73      1316

Base Validation Confusion Matrix:
 [[199 213]
 [129 775]]
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best params: {'classifier__subsample': 1.0, 'classifier__n_estimators': 300, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.1, 'classifier__gamma': 1, 'classifier__colsample_bytree': 0.6}
Best CV score: 0.6885451264010513
Tuned Validation Accuracy: 0.741641337

In [16]:
import joblib
from pathlib import Path

out_dir = Path("models")
out_dir.mkdir(exist_ok=True)

artifact_map = {
    "best_model_put.joblib": best_model,
    "model_xgb_put.joblib": model_xgb,
    "label_encoder_put.joblib": le,
}

saved = []
for fname, obj in artifact_map.items():
    p = out_dir / fname
    joblib.dump(obj, p)
    saved.append(str(p))

print("Saved files:", saved)



Saved files: ['models\\best_model_put.joblib', 'models\\model_xgb_put.joblib', 'models\\label_encoder_put.joblib']


In [38]:
# Feature significance (Permutation Importance on validation split)
from sklearn.inspection import permutation_importance
import pandas as pd

if "best_model" not in globals():
    raise ValueError("best_model not found. Run training cells first.")
if "X_val" not in globals() or "y_val_enc" not in globals():
    raise ValueError("X_val / y_val_enc not found. Run split+training cell first.")

perm = permutation_importance(
    best_model,
    X_val,
    y_val_enc,
    n_repeats=10,
    random_state=42,
    scoring="accuracy",
    n_jobs=-1,
)

importance_df = pd.DataFrame({
    "feature": X_val.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

print("Top 25 significant features (validation permutation importance):")
display(importance_df.head(25))



Top 25 significant features (validation permutation importance):


,feature,importance_mean,importance_std
8,Put_Option_Price,0.149490,0.010378
11,todays_range,0.013095,0.003844
4,Average_True_Range,0.011139,0.003811
7,Call_Option_Price,0.006633,0.003289
9,strike_to_close_price_gap,0.005952,0.005524
13,days_to_options_expiry,0.002976,0.003552
1,Performance_Week,0.002551,0.004166
5,Change,0.002381,0.002041
16,Calls_OpenInterest,0.001616,0.004015
0,Stock_Price,0.001105,0.003746
